In [ ]:
#setup for processed data
#Note: use for BinaryArray data produced from Entrainment_Preprocessing.ipynb
def GetProcessedString(PROCESSING=False):
    if PROCESSING==True:
        Processed_string="PROCESSED_"
    else:
        Processed_string=""
    return Processed_string

PROCESSING=False 
PROCESSING=True #set to True if using Turbulence-Removed Binary Arrays
Processed_string = GetProcessedString(PROCESSING=PROCESSING)

In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from FUNCTIONS_Variable_Calculation import *

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=2)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="EntrainmentCalculation", dataName="EntrainmentCalculation_DivideMassFlux",
                                dtype='int32')

In [25]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

Running timesteps from 0:6 



In [26]:
####################################
#FUNCTIONS

In [27]:
#CALCULATING ENTRAINMENT
def GetVariable_T(t, varName, cache=None): #*MassFlux
    """Load variable for a given timestep, using cached open HDF5 files if available."""
    if cache is None:
        cache = {}

    timeString = ModelData.timeStrings[t]

    # If file for this timestep already in cache, reuse it
    if timeString in cache:
        f = cache[timeString]
    else:
        # Otherwise, open new file and cache it
        f = DataManager.GetTimestepData_V2(DataManager.inputDataDirectory, timeString)
        cache[timeString] = f

    # Load the desired variable from the open file (lazy read)
    if varName in f.keys():
        output = f[varName][:]  # read actual data
    else:
        # fallback for derived variables
        output = CallVariable(ModelData, DataManager, timeString, variableName=varName)
    return output

def SaveDivide(numerator, denominator):
    # 1. Create an array of NaNs with the same shape as the numerator
    result = np.full(numerator.shape, np.nan, dtype=np.float32)
    
    # 2. Find where we can safely divide
    nonzero = (denominator != 0)
    
    # 3. Only fill the "nonzero" indices with the division result
    result[nonzero] = numerator[nonzero] / denominator[nonzero]
    
    return result
    
def LoadMassFlux(t): #*MassFlux
    
    updraftType = "g" 
    massFluxVarName = f"{Processed_string}MassFlux_{updraftType}"
    MassFlux_g = GetVariable_T(t=t, varName=massFluxVarName, cache=None)

    updraftType = "c" 
    massFluxVarName = f"{Processed_string}MassFlux_{updraftType}"
    MassFlux_c = GetVariable_T(t=t, varName=massFluxVarName, cache=None)

    return MassFlux_g, MassFlux_c

def DivideByMassFlux(t, varName,
                     MassFlux_g, MassFlux_c):
    variable = GetVariable_T(t, varName)
    MassFlux = MassFlux_g if "g" in varName else MassFlux_c
    variable_DivideMassFlux = SaveDivide(variable, MassFlux)
    return variable_DivideMassFlux

In [28]:
def GetVarNames_Entrainment(Processed_string):
    varNames = [f"{Processed_string}Entrainment_g",
                f"{Processed_string}Entrainment_c"]
    
    varNames += [f"{Processed_string}TransferEntrainment_g",
                f"{Processed_string}TransferEntrainment_c"]
    return varNames

def GetVarNames_Detrainment(Processed_string):
    varNames = [f"{Processed_string}Detrainment_g",
                f"{Processed_string}Detrainment_c"]
    
    varNames += [f"{Processed_string}TransferDetrainment_g",
                f"{Processed_string}TransferDetrainment_c"]
    return varNames

def AddDivideMassFluxString(varName):
    parts = varName.rsplit('_', 1)
    varName_DivideMassFlux = f"{parts[0]}_DivideMassFlux_{parts[1]}"
    return varName_DivideMassFlux

In [29]:
def RunCalculation_Entrainment(t, Processed_string,
                               MassFlux_g, MassFlux_c): 

    varNames = GetVarNames_Entrainment(Processed_string)

    outputDictionary_Variable_DivideMassFlux = {}
    for varName in varNames:
        variable_DivideMassFlux = DivideByMassFlux(t, varName,
                                                   MassFlux_g, MassFlux_c)

        varName_DivideMassFlux = AddDivideMassFluxString(varName)
        outputDictionary_Variable_DivideMassFlux[varName_DivideMassFlux] = variable_DivideMassFlux
        
    return outputDictionary_Variable_DivideMassFlux

def RunCalculation_Detrainment(t, Processed_string,
                               MassFlux_g, MassFlux_c): 

    varNames = GetVarNames_Detrainment(Processed_string)

    outputDictionary_Variable_DivideMassFlux = {}
    for varName in varNames:
        variable_DivideMassFlux = DivideByMassFlux(t, varName,
                                                   MassFlux_g, MassFlux_c)

        varName_DivideMassFlux = AddDivideMassFluxString(varName)
        outputDictionary_Variable_DivideMassFlux[varName_DivideMassFlux] = variable_DivideMassFlux
        
    return outputDictionary_Variable_DivideMassFlux

In [30]:
##############################################
#RUNNING

In [ ]:
#running calculation
for t in tqdm(loop_elements, total=len(loop_elements)):
    # if np.mod(t,1)==0: print(f'Current time {t}')
    if t == ModelData.Ntime-1:
        continue

    #loading MassFlux
    [MassFlux_g, MassFlux_c] =  LoadMassFlux(t)

    #calculating variables
    outputDictionary_Entrainment_DivideMassFlux = RunCalculation_Entrainment(t, Processed_string,
                                                                         MassFlux_g, MassFlux_c)
    outputDictionary_Detrainment_DivideMassFlux = RunCalculation_Detrainment(t, Processed_string,
                                                                             MassFlux_g, MassFlux_c)
    
    #outputting
    timeString = ModelData.timeStrings[t]
    
    DataManager.SaveOutputTimestep(DataManager.outputDataDirectory, timeString, outputDictionary_Entrainment_DivideMassFlux, dataName=f"{Processed_string}Entrainment_DivideMassFlux",dtype='float32')

    DataManager.SaveOutputTimestep(DataManager.outputDataDirectory, timeString, outputDictionary_Detrainment_DivideMassFlux, dataName=f"{Processed_string}Detrainment_DivideMassFlux",dtype='float32')